# Sesión 1 · La plataforma de datos desde cero
**Databricks AI Engineer** — caso Neptuno

Al terminar este notebook vas a tener:
- tu propio catálogo gobernado en Unity Catalog,
- las 8 tablas de Neptuno en la capa **bronze**,
- y la red de seguridad de Delta funcionando (time travel).

> 📏 **Regla de la casa:** bronze no se corrige, se re-procesa.

## 0 · Tu identidad en el curso
Cada alumno trabaja en su propio catálogo para no pisarse.

In [0]:
import re, unicodedata

dbutils.widgets.text("alumno", "Emerson Suarez", "Tu nombre")
crudo = dbutils.widgets.get("alumno").strip()
assert crudo, "Escribe tu nombre en el widget de arriba antes de seguir."

def sanear(nombre: str) -> str:
    """Un catálogo de Unity Catalog no admite espacios ni tildes.
    'Demo Testing' -> 'demo_testing' · 'José Pérez' -> 'jose_perez'"""
    sin_tildes = unicodedata.normalize("NFKD", nombre).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-zA-Z0-9]+", "_", sin_tildes).strip("_").lower()

ALUMNO = sanear(crudo)
assert ALUMNO, "Pon tu nombre en el widget de arriba antes de seguir."

CATALOGO = f"neptuno_{ALUMNO}"
print(f"Hola, {crudo}.\nTu catálogo: {CATALOGO}")

Hola, Emerson Suarez.
Tu catálogo: neptuno_emerson_suarez


## 1 · Crear el catálogo, los esquemas y el Volume
La jerarquía de Unity Catalog: `metastore → catálogo → esquema → tabla | volume`.

El **Volume** es almacenamiento de archivos *gobernado*: los CSV de hoy y los
documentos de la sesión 4 (RAG) van a vivir ahí.

In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOGO}")
for capa in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{capa}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.bronze.landing")

LANDING = f"/Volumes/{CATALOGO}/bronze/landing"
print("Landing zone:", LANDING)
display(spark.sql(f"SHOW SCHEMAS IN {CATALOGO}"))

Landing zone: /Volumes/neptuno_emerson_suarez/bronze/landing


databaseName
bronze
default
gold
information_schema
silver


## 2 · Traer los CSV de Neptuno a tu Volume
Los datos del curso viven en un volumen compartido, en modo lectura.
Esta celda los copia al tuyo; si ya están, no hace nada.

In [0]:
COMPARTIDO = "/Volumes/neptuno/ventas/landing"
ESPERADOS = ["categorias", "clientes", "detalles_pedidos", "empleados",
             "pedidos", "productos", "proveedores", "transportistas"]

ya_estan = {f.name for f in dbutils.fs.ls(LANDING)}
copiados = 0
for tabla in ESPERADOS:
    if f"{tabla}.csv" not in ya_estan:
        dbutils.fs.cp(f"{COMPARTIDO}/{tabla}.csv", f"{LANDING}/{tabla}.csv")
        copiados += 1
print(f"{copiados} archivos copiados · {len(ESPERADOS)-copiados} ya estaban")

0 archivos copiados · 8 ya estaban


### Verificación: ¿están los ocho?

In [0]:
archivos = [f.name for f in dbutils.fs.ls(LANDING)]
print(f"{len(archivos)} archivos en el landing:")
for a in sorted(archivos):
    print(" ·", a)

faltan = [t for t in ESPERADOS if not any(t in a for a in archivos)]
assert not faltan, f"Faltan estos CSV en el landing: {faltan}"
print("\n✅ están los 8")

8 archivos en el landing:
 · categorias.csv
 · clientes.csv
 · detalles_pedidos.csv
 · empleados.csv
 · pedidos.csv
 · productos.csv
 · proveedores.csv
 · transportistas.csv

✅ están los 8


## 3 · Bronze: cargar tal como llegó
Bronze guarda el dato **crudo**, más metadata de ingesta: de qué archivo vino y cuándo.
Esa metadata es lo que después permite responder «¿de dónde salió este número?».

In [0]:
from pyspark.sql import functions as F

for tabla in ESPERADOS:
    (spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{LANDING}/{tabla}.csv")
        # _metadata.file_path y no input_file_name(): esta última no está
        # permitida en Unity Catalog con acceso compartido.
        .withColumn("_archivo_origen", F.col("_metadata.file_path"))
        .withColumn("_ingesta_ts",    F.current_timestamp())
        .write.mode("overwrite")
        .saveAsTable(f"{CATALOGO}.bronze.{tabla}"))
    n = spark.table(f"{CATALOGO}.bronze.{tabla}").count()
    print(f"bronze.{tabla:20s} {n:>6,} filas")

bronze.categorias                8 filas
bronze.clientes                 91 filas
bronze.detalles_pedidos      2,155 filas
bronze.empleados                 9 filas
bronze.pedidos                 830 filas
bronze.productos                77 filas
bronze.proveedores              29 filas
bronze.transportistas            6 filas


### Verificación por efecto, no por fe
No alcanza con que el bloque no haya tirado error: hay que **contar**.

In [0]:
CONTEOS_ESPERADOS = {"categorias": 8, "clientes": 91, "detalles_pedidos": 2155,
                     "empleados": 9, "pedidos": 830, "productos": 77,
                     "proveedores": 29, "transportistas": 6}

errores = []
for tabla, esperado in CONTEOS_ESPERADOS.items():
    real = spark.table(f"{CATALOGO}.bronze.{tabla}").count()
    estado = "✅" if real == esperado else "❌"
    if real != esperado:
        errores.append((tabla, esperado, real))
    print(f"{estado} {tabla:20s} esperado {esperado:>6,} · real {real:>6,}")

assert not errores, f"Conteos que no cuadran: {errores}"
print("\n🎉 Neptuno está en tu bronze.")

✅ categorias           esperado      8 · real      8
✅ clientes             esperado     91 · real     91
✅ detalles_pedidos     esperado  2,155 · real  2,155
✅ empleados            esperado      9 · real      9
✅ pedidos              esperado    830 · real    830
✅ productos            esperado     77 · real     77
✅ proveedores          esperado     29 · real     29
✅ transportistas       esperado      6 · real      6

🎉 Neptuno está en tu bronze.


## 4 · Delta: la red de seguridad
Delta Lake es **Parquet + un log de transacciones**. De ese log sale todo lo demás:
transacciones ACID, versionado y *time travel*.

In [0]:
spark.sql(f"DESCRIBE HISTORY {CATALOGO}.bronze.productos").display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
3,2026-08-31T04:42:13.000Z,74220838578198,emersonsuarez2904@gmail.com,RESTORE,"Map(version -> 0, timestamp -> null)",null,List(63612918197214),c5f2c2b1-0211-417b-8a60-f79ab43531f8,0831-035245-dqogve4l-v2n,2,Serializable,false,"Map(numRestoredFiles -> 1, removedFilesSize -> 6115, numRemovedFiles -> 1, restoredFilesSize -> 6367, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 6367)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
2,2026-08-31T04:41:52.000Z,74220838578198,emersonsuarez2904@gmail.com,OPTIMIZE,"Map(predicate -> [], auto -> true, clusterBy -> [], zOrderBy -> [], batchId -> 0)",null,List(63612918197214),7529c3e8-556d-425b-8b5f-b60c258e28ca,0831-035245-dqogve4l-v2n,1,SnapshotIsolation,false,"Map(numRemovedFiles -> 1, numRemovedBytes -> 6367, p25FileSize -> 6115, numDeletionVectorsRemoved -> 1, minFileSize -> 6115, numAddedFiles -> 1, maxFileSize -> 6115, p75FileSize -> 6115, p50FileSize -> 6115, numAddedBytes -> 6115)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
1,2026-08-31T04:41:49.000Z,74220838578198,emersonsuarez2904@gmail.com,DELETE,"Map(predicate -> [""(IdCategoria#17186 = 1)""])",null,List(63612918197214),7529c3e8-556d-425b-8b5f-b60c258e28ca,0831-035245-dqogve4l-v2n,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 3130, numDeletionVectorsUpdated -> 0, numDeletedRows -> 12, scanTimeMs -> 2140, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 968)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13
0,2026-08-31T04:13:55.000Z,74220838578198,emersonsuarez2904@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(63612918197214),88d61de6-afaa-4fc8-8b4b-9366c85db2ae,0831-035245-dqogve4l-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 77, numOutputBytes -> 6367)",null,Databricks-Runtime/19.2.x-aarch64-photon-scala2.13


### El accidente (a propósito)
Alguien borra todas las bebidas. Sin backup.

In [0]:
# Guardamos la versión actual: si el notebook ya se corrió antes, la versión 0
# no es la carga limpia de HOY, y restaurar a 0 traería otro estado.
version_buena = (spark.sql(f"DESCRIBE HISTORY {CATALOGO}.bronze.productos")
                      .selectExpr("max(version) AS v").first()["v"])
antes = spark.table(f"{CATALOGO}.bronze.productos").count()
print(f"versión buena: {version_buena} · {antes} productos")
spark.sql(f"DELETE FROM {CATALOGO}.bronze.productos WHERE IdCategoria = 1")
despues = spark.table(f"{CATALOGO}.bronze.productos").count()
print(f"antes: {antes} · después del DELETE: {despues} · se fueron {antes - despues}")

versión buena: 0 · 77 productos
antes: 77 · después del DELETE: 65 · se fueron 12


### Y vuelve

In [0]:
import time

def restaurar(tabla: str, version: int, intentos: int = 4):
    """Databricks puede lanzar un OPTIMIZE automático justo entre el borrado y la
    restauración. Cuando eso pasa, el RESTORE choca con esa transacción y falla con
    ConcurrentWriteException. No es un error nuestro: se reintenta y entra."""
    for i in range(1, intentos + 1):
        try:
            spark.sql(f"RESTORE TABLE {tabla} TO VERSION AS OF {version}")
            if i > 1:
                print(f"   (entró en el intento {i}: había una escritura automática en curso)")
            return True
        except Exception as err:
            if "CONCURRENT" not in str(err).upper() or i == intentos:
                raise
            time.sleep(2 * i)
    return False

restaurar(f"{CATALOGO}.bronze.productos", version_buena)
recuperado = spark.table(f"{CATALOGO}.bronze.productos").count()
print(f"después del RESTORE: {recuperado}")
assert recuperado == antes, "El restore no devolvió todas las filas"
print("✅ sin backup, sin drama")

después del RESTORE: 77
✅ sin backup, sin drama


## 5 · La trampa del descuento
Siembra para la sesión 2. **No la resolvemos hoy.**

El ingreso de una línea **no** es `PrecioUnidad * Cantidad`: hay un `Descuento`
que hay que restar. La diferencia es chica — y por eso llega a producción.

In [0]:
spark.sql(f"""
SELECT ROUND(SUM(PrecioUnidad * Cantidad), 2)                   AS ingreso_ingenuo,
       ROUND(SUM(PrecioUnidad * Cantidad * (1 - Descuento)), 2) AS ingreso_real,
       ROUND(100 * (1 - SUM(PrecioUnidad * Cantidad * (1 - Descuento))
                      / SUM(PrecioUnidad * Cantidad)), 2)       AS pct_de_error
FROM {CATALOGO}.bronze.detalles_pedidos
""").display()

ingreso_ingenuo,ingreso_real,pct_de_error
1354458.59,1265793.04,6.55


> **¿Cuánto da `pct_de_error`?** Ese número es el motivo por el que este curso existe.
> Nadie audita un error de esa magnitud. Y una IA que lee esta tabla sin saber que
> el descuento existe, lo va a cometer todas las veces.

pct_de_error igual a 6.55

## 6 · Tu entregable
1. Las 8 tablas en `bronze` ✅ (ya está)
2. Tres consultas SQL que respondan preguntas de negocio reales
3. Una tabla documentada con `COMMENT ON TABLE`
4. Un cambio provocado y revertido con `RESTORE` ✅ (ya está)

### 6.2 · Tres consultas de negocio
Estas tres están resueltas para que veas el patrón. La cuarta es tuya.

**1 · ¿Qué categoría vende más?** Fíjate que el ingreso lleva el descuento aplicado.

In [0]:
spark.sql(f"""
SELECT c.NombreCategoria                                             AS categoria,
       ROUND(SUM(d.PrecioUnidad * d.Cantidad * (1 - d.Descuento)), 2) AS ingreso_neto,
       SUM(d.Cantidad)                                                AS unidades
FROM {CATALOGO}.bronze.detalles_pedidos d
JOIN {CATALOGO}.bronze.productos  p ON d.IdProducto  = p.IdProducto
JOIN {CATALOGO}.bronze.categorias c ON p.IdCategoria = c.IdCategoria
GROUP BY 1 ORDER BY ingreso_neto DESC
""").display()

categoria,ingreso_neto,unidades
Bebidas,267868.18,9532
Lacteos,234507.29,9149
Reposteria,167357.22,7906
Carnes y Aves,163022.36,4199
Pescados y Mariscos,131261.74,7681
Condimentos,106047.09,5298
Frutas y Verduras,99984.58,2990
Granos y Cereales,95744.59,4562


> 🔍 **Mira el número de Bebidas.** Es el mismo que el Genie del final de la clase
> va a llamar «margen». No es un margen: es la venta neta, con otro nombre.
> El margen necesitaría costos, y en Neptuno no hay tabla de costos.

**2 · ¿Qué cliente hace más pedidos?**

In [0]:
spark.sql(f"""
SELECT c.NombreCompania AS cliente, c.Pais, COUNT(*) AS pedidos
FROM {CATALOGO}.bronze.pedidos   p
JOIN {CATALOGO}.bronze.clientes  c ON p.IdCliente = c.IdCliente
GROUP BY 1, 2 ORDER BY pedidos DESC LIMIT 10
""").display()

cliente,Pais,pedidos
Save-a-lot Markets,Estados Unidos,31
Ernst Handel,Austria,30
QUICK-Stop,Alemania,28
Hungry Owl All-Night Grocers,Irlanda,19
Folk och fä HB,Suecia,19
Rattlesnake Canyon Grocery,Estados Unidos,18
HILARION-Abastos,Venezuela,18
Berglunds snabbköp,Suecia,18
Bon app',Francia,17
Lehmanns Marktstand,Alemania,15


**3 · ¿Qué transportista despacha más lento?**
Días entre el pedido y el envío. Ojo con las tres fechas: `FechaPedido` es cuándo se
vendió, `FechaEnvio` cuándo salió, y `FechaEntrega` es la fecha *comprometida*, no la real.

In [0]:
spark.sql(f"""
SELECT t.NombreCompania AS transportista,
       COUNT(*)                                                                   AS pedidos,
       ROUND(AVG(DATEDIFF(TO_DATE(p.FechaEnvio), TO_DATE(p.FechaPedido))), 1)     AS dias_promedio,
       MAX(DATEDIFF(TO_DATE(p.FechaEnvio), TO_DATE(p.FechaPedido)))               AS peor_caso
FROM {CATALOGO}.bronze.pedidos         p
JOIN {CATALOGO}.bronze.transportistas  t ON p.IdTransportista = t.IdTransportista
WHERE p.FechaEnvio IS NOT NULL
GROUP BY 1 ORDER BY dias_promedio DESC
""").display()

transportista,pedidos,dias_promedio,peor_caso
Paquetes Unidos,315,9.2,37
Expreso Veloz,245,8.6,37
Envios Federales,249,7.5,35


> 😏 Fíjate en los nombres frente a los números. El que más tarda y el que se llama
> «Expreso Veloz» no coinciden con lo que uno esperaría.

**4 · La tuya.** Escribe una consulta que responda algo que a ti te interese.

In [0]:
### ¿Cuáles son los 10 productos que generan mayor ingreso neto?
spark.sql(f"""
SELECT p.NombreProducto AS producto,
       ROUND(SUM(d.PrecioUnidad * d.Cantidad * (1 - d.Descuento)), 2) AS ingreso_neto,
       SUM(d.Cantidad) AS unidades_vendidas
FROM {CATALOGO}.bronze.detalles_pedidos d
JOIN {CATALOGO}.bronze.productos p
  ON d.IdProducto = p.IdProducto
GROUP BY p.NombreProducto
ORDER BY ingreso_neto DESC
LIMIT 10
""").display()



producto,ingreso_neto,unidades_vendidas
Côte de Blaye,141396.74,623
Thüringer Rostbratwurst,80368.67,746
Raclette Courdavault,71155.7,1496
Tarte au sucre,47234.97,1083
Camembert Pierrot,46825.48,1577
Gnocchi di nonna Alice,42593.06,1263
Manjimup Dried Apples,41819.65,886
Alice Mutton,32698.38,978
Carnarvon Tigers,29171.88,539
Rössle Sauerkraut,25696.64,640


### 6.3 · Documenta una tabla
Escribe lo que un colega nuevo necesitaría saber para **no** equivocarse con ella.
En la sesión 3 vas a descubrir que esto es exactamente lo que le falta al Genie del Demo 0.

In [0]:
spark.sql(f"""
COMMENT ON TABLE {CATALOGO}.bronze.detalles_pedidos IS
'Tabla de detalle de pedidos. Una fila representa un producto dentro de un pedido. Para calcular el ingreso neto se debe usar PrecioUnidad * Cantidad * (1 - Descuento); Descuento es una proporción entre 0 y 1. _archivo_origen y _ingesta_ts identifican la procedencia y el momento de carga.'
""")

DataFrame[]

In [0]:
spark.sql(f"DESCRIBE TABLE EXTENDED {CATALOGO}.bronze.detalles_pedidos").display()

col_name,data_type,comment
IdPedido,int,null
IdProducto,int,null
PrecioUnidad,double,null
Cantidad,int,null
Descuento,double,null
_archivo_origen,string,null
_ingesta_ts,timestamp,null
,,
# Delta Statistics Columns,,
Column Names,"_ingesta_ts, Descuento, Cantidad, IdProducto, IdPedido, _archivo_origen, PrecioUnidad",


### Verificación final
Si la ejecución se cortó entre el borrado y la restauración, tus datos quedaron
incompletos. Esta celda lo detecta antes de que te vayas.

In [0]:
def verificacion_final():
    problemas = []
    for tabla, esperado in CONTEOS_ESPERADOS.items():
        real = spark.table(f"{CATALOGO}.bronze.{tabla}").count()
        if real != esperado:
            problemas.append(f"{tabla}: {real} filas, deberían ser {esperado}")
    if problemas:
        print("⚠️  Tus datos quedaron incompletos:")
        for p in problemas: print("   ·", p)
        print("\n   Vuelve a correr la celda de carga (sección 3) para dejarlos bien.")
    else:
        print("✅ Las 8 tablas están completas. Puedes cerrar tranquilo.")
    return not problemas

verificacion_final()

✅ Las 8 tablas están completas. Puedes cerrar tranquilo.


True

---
## Antes de irte
**Apaga el compute.** Un cluster olvidado encendido es la forma más cara de no aprender nada.

**La semana que viene:** hoy cargamos todo de una vez. En la vida real los datos llegan de a
poco, todos los días. En la sesión 2 el pipeline se entera solo de lo que llegó nuevo.